# Análise Exploratória (EDA) — Performance de Ações na B3 (2021–2023)

**Projeto Final — Visualização de Dados e BI | Linha 2: Mercado de Ações (B3)**
**Autor:** André Abranjo Ramos

## Objetivo
Explorar a base tratada pelo pipeline (`src/`) **antes** de construir o Data App, para:
1. conhecer a estrutura e a qualidade dos dados;
2. responder, em primeira versão, às **5 perguntas de negócio** do edital;
3. prototipar os gráficos em **Plotly Express** que irão para o Streamlit.

## Perguntas de negócio (edital, Linha 2)
| # | Pergunta | Visual |
|---|---|---|
| P1 | Rentabilidade comparativa **Base 100** de 2021 a 2023 | Linhas normalizadas |
| P2 | Melhor relação **risco × retorno** (Sharpe empírico) | Dispersão |
| P3 | Comportamento estatístico dos **retornos diários** | Histograma + boxplot |
| P4 | **Drawdown máximo** e ativo mais resiliente | Área de drawdown |
| P5 | Preço frente às **médias móveis** SMA 21 e SMA 50 | Multilinha |

> **Cuidado metodológico:** esta análise é **descritiva**. Correlação e desempenho passado **não implicam causalidade** nem garantem resultados futuros.

**Como executar:** abra no VS Code, selecione o kernel da `.venv` e use *Run All*. O pipeline (`python -m src.extract`, `src.eventos`, `src.transform`, `src.metrics`) deve ter sido executado antes.

## 1. Configuração do ambiente

In [1]:
# Bibliotecas
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

# Localiza a RAIZ do projeto subindo pastas até encontrar "src" (caminho relativo,
# funciona em qualquer computador - item 4.3 do edital)
RAIZ = Path.cwd()
while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src import config  # parâmetros centrais do projeto

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 30)
TEMA = "plotly_white"                      # tema claro, alto contraste
print("Raiz do projeto:", RAIZ)

Raiz do projeto: c:\Users\k34404\Downloads\SCTEC\Projeto_final\Projeto


## 2. Carga dos dados tratados
Três arquivos gerados pelo pipeline: a **série diária** com indicadores, o **resumo por ativo** e o **relatório de qualidade**.

In [2]:
diario = pd.read_csv(config.DIR_DADOS_TRATADOS / "cotacoes_tratadas.csv", parse_dates=["data"])
resumo = pd.read_csv(config.DIR_DADOS_TRATADOS / "resumo_ativos.csv", parse_dates=["data_inicio", "data_fim", "data_drawdown_maximo"])
qualidade = pd.read_csv(config.DIR_DADOS_TRATADOS / "relatorio_limpeza.csv")

acoes = diario[diario["codigo"] != "IBOV"]          # carteira de 12 ações
ibov = diario[diario["codigo"] == "IBOV"]            # benchmark

print(f"Série diária : {diario.shape[0]:,} linhas x {diario.shape[1]} colunas")
print(f"Ativos       : {diario['codigo'].nunique()} (12 ações + Ibovespa)")
print(f"Período      : {diario['data'].min():%d/%m/%Y} a {diario['data'].max():%d/%m/%Y}")
print(f"Pregões/ativo: {diario.groupby('codigo').size().unique()}")

Série diária : 9,685 linhas x 22 colunas
Ativos       : 13 (12 ações + Ibovespa)
Período      : 04/01/2021 a 28/12/2023
Pregões/ativo: [745]


### 2.1 Estrutura e tipos

In [3]:
diario.dtypes.to_frame("tipo").T

,data,abertura,maxima,minima,fechamento,fechamento_ajustado,volume,ticker,ticker_fonte,codigo,pregao_sintetico,fator_evento,retorno_diario,base_100,retorno_acumulado,volume_financeiro,pico_historico,drawdown,sma_21,sma_50,empresa,setor
tipo,datetime64[us],float64,float64,float64,float64,float64,int64,str,str,str,bool,float64,float64,float64,float64,float64,float64,float64,float64,float64,str,str


In [4]:
diario.head()

,data,abertura,maxima,minima,fechamento,fechamento_ajustado,volume,ticker,ticker_fonte,codigo,pregao_sintetico,fator_evento,retorno_diario,base_100,retorno_acumulado,volume_financeiro,pico_historico,drawdown,sma_21,sma_50,empresa,setor
0,2021-01-04,15.7900,15.8500,15.3800,15.4800,11.7161,17835200,ABEV3.SA,ABEV3.SA,ABEV3,False,1.0000,NaN,100.0000,0.0000,"276,088,887.8357",11.7161,0.0000,NaN,NaN,Ambev,Consumo
1,2021-01-05,15.3700,16.0200,15.2300,15.8100,11.9659,37105800,ABEV3.SA,ABEV3.SA,ABEV3,False,1.0000,0.0213,102.1318,0.0213,"586,642,713.5702",11.9659,0.0000,NaN,NaN,Ambev,Consumo
2,2021-01-06,15.6600,16.2300,15.6300,16.1300,12.2081,28371600,ABEV3.SA,ABEV3.SA,ABEV3,False,1.0000,0.0202,104.1990,0.0420,"457,633,884.1896",12.2081,0.0000,NaN,NaN,Ambev,Consumo
3,2021-01-07,16.0300,16.2400,15.9100,16.0500,12.1475,32540100,ABEV3.SA,ABEV3.SA,ABEV3,False,1.0000,-0.0050,103.6822,0.0368,"522,268,580.1739",12.2081,-0.0050,NaN,NaN,Ambev,Consumo
4,2021-01-08,16.1000,16.7500,16.0300,16.5300,12.5108,30188800,ABEV3.SA,ABEV3.SA,ABEV3,False,1.0000,0.0299,106.7830,0.0678,"499,020,884.7290",12.5108,0.0000,NaN,NaN,Ambev,Consumo


### 2.2 Qualidade dos dados (evidência do pipeline)
Resumo das verificações executadas em `src/transform.py`. Linhas com ocorrência maior que zero foram **tratadas ou documentadas**.

In [5]:
qualidade[["etapa", "verificacao", "ocorrencias", "acao"]]

,etapa,verificacao,ocorrencias,acao
0,encoding,Leitura em UTF-8,0,arquivo lido sem erro de encoding
1,padronizacao,Colunas renomeadas para português (snake_case),0,9 colunas padronizadas + coluna 'codigo'
2,tipos,Valores não numéricos em preços/volume,0,convertidos para float64 (preços); inválidos v...
3,datas,Datas ilegíveis ou vazias,0,todas convertidas para datetime64 (sem fuso ho...
4,duplicados,"Linhas repetidas por (ticker, data)",0,nenhuma remoção necessária
5,nulos,Preços nulos,0,nenhum tratamento necessário
6,nulos,Volumes nulos,0,nenhum tratamento necessário
7,nulos,Linhas ainda sem preço ajustado após o preench...,0,nenhuma remoção necessária
8,consistencia,Preços menores ou iguais a zero,0,nenhuma remoção necessária
9,consistencia,Máxima menor que a mínima,0,ok


In [6]:
# Ocorrências relevantes da fonte
print("Pregões sintéticos (preço repetido e volume zero):")
print(diario.loc[diario["pregao_sintetico"], ["data", "codigo", "fechamento", "volume"]].to_string(index=False))
print("\nEventos societários ajustados:")
print(pd.read_csv(config.ARQ_EVENTOS)[["ticker", "data_ex", "ativo_recebido", "fator_ajuste"]].to_string(index=False))

Pregões sintéticos (preço repetido e volume zero):
      data codigo  fechamento  volume
2023-06-02  BBDC4     15.7800       0
2022-09-26  PRIO3     28.1200       0
2022-09-26  WEGE3     30.5700       0

Eventos societários ajustados:
  ticker    data_ex ativo_recebido  fator_ajuste
ITUB4.SA 2021-10-04      XPBR31.SA        0.8258


### 2.3 Estatísticas descritivas dos retornos diários (%)

In [7]:
(acoes.pivot(index="data", columns="codigo", values="retorno_diario") * 100).describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
codigo,,,,,,,,
ABEV3,744.0000,0.0200,1.6000,-5.7000,-0.8800,0.0000,0.8900,9.7200
BBDC4,744.0000,0.0100,1.9800,-17.3800,-1.0300,0.0000,1.1300,6.9100
EGIE3,744.0000,0.0400,1.1800,-4.7500,-0.6500,0.0300,0.7300,5.4000
EMBR3,744.0000,0.1800,2.9500,-14.9300,-1.3800,0.0000,1.7000,16.0200
GGBR4,744.0000,0.0600,2.2400,-7.9300,-1.2800,0.0400,1.3300,9.0000
ITUB4,744.0000,0.0700,1.6900,-7.2800,-1.0200,0.0000,1.1200,8.2700
KLBN11,744.0000,0.0100,1.6800,-6.2500,-1.0600,-0.0400,1.0300,7.5600
PETR4,744.0000,0.1900,2.4800,-20.7100,-1.0500,0.1700,1.5800,11.0300
PRIO3,744.0000,0.1900,2.9500,-10.7600,-1.6000,0.1500,1.8900,17.8700


## 3. P1 — Rentabilidade comparativa (Base 100)
*Se um investidor aplicasse R$ 100 em cada ativo no início de 2021, quanto teria ao final de 2023?*

In [8]:
fig = px.line(
    diario, x="data", y="base_100", color="codigo",
    title="P1 · Desempenho normalizado (Base 100) — 2021 a 2023",
    labels={"data": "", "base_100": "Base 100 (R$ 100 investidos)", "codigo": "Ativo"},
    template=TEMA,
)
fig.add_hline(y=100, line_dash="dash", line_color="gray", annotation_text="Nível inicial (100)")
fig.update_traces(selector=dict(name="IBOV"), line=dict(width=4, color="black", dash="dot"))
fig.show()

In [9]:
ranking = resumo.sort_values("retorno_acumulado", ascending=False)[["codigo", "setor", "retorno_acumulado"]]
ret_ibov = resumo.loc[resumo["codigo"] == "IBOV", "retorno_acumulado"].iloc[0]
acima = ranking[(ranking["codigo"] != "IBOV") & (ranking["retorno_acumulado"] > ret_ibov)]
print(f"Ibovespa no período: {ret_ibov:+.1%}")
print(f"Ações acima do Ibovespa: {len(acima)} de 12 -> {', '.join(acima['codigo'])}")
ranking.assign(retorno_acumulado=lambda d: (d["retorno_acumulado"] * 100).round(1)).rename(
    columns={"retorno_acumulado": "retorno_acumulado_%"}).reset_index(drop=True)

Ibovespa no período: +13.2%
Ações acima do Ibovespa: 7 de 12 -> PETR4, PRIO3, EMBR3, ITUB4, GGBR4, EGIE3, VALE3


,codigo,setor,retorno_acumulado_%
0,PETR4,Energia,228.7000
1,PRIO3,Energia,208.3000
2,EMBR3,Indústria & Materiais,167.5000
3,ITUB4,Financeiro,50.2000
4,GGBR4,Indústria & Materiais,30.8000
5,EGIE3,Energia,25.3000
6,VALE3,Indústria & Materiais,18.1000
7,IBOV,Benchmark,13.2000
8,WEGE3,Indústria & Materiais,3.5000
9,ABEV3,Consumo,2.6000


## 4. P2 — Risco × Retorno (Sharpe empírico)
*Quais ativos entregaram mais retorno por unidade de risco?*
Sharpe empírico = retorno anualizado ÷ volatilidade anualizada (taxa livre de risco = 0, simplificação documentada).

In [10]:
dados_p2 = resumo.assign(
    vol_pct=resumo["volatilidade_anual"] * 100,
    ret_pct=resumo["retorno_acumulado"] * 100,
    tamanho=resumo["volume_financeiro_medio"].fillna(resumo["volume_financeiro_medio"].median()),
)
fig = px.scatter(
    dados_p2, x="vol_pct", y="ret_pct", size="tamanho", color="setor", text="codigo",
    title="P2 · Risco x Retorno — tamanho da bolha = volume financeiro médio diário",
    labels={"vol_pct": "Volatilidade anualizada (% a.a.)", "ret_pct": "Retorno acumulado 2021-2023 (%)", "setor": "Setor"},
    template=TEMA, size_max=45,
)
fig.update_traces(textposition="top center")
fig.add_hline(y=ret_ibov * 100, line_dash="dash", line_color="gray", annotation_text="Retorno do Ibovespa")
fig.show()

In [11]:
resumo[["codigo", "setor", "retorno_anualizado", "volatilidade_anual", "sharpe_empirico"]].sort_values(
    "sharpe_empirico", ascending=False).round(3).reset_index(drop=True)

,codigo,setor,retorno_anualizado,volatilidade_anual,sharpe_empirico
0,PETR4,Energia,0.4960,0.3940,1.2590
1,PRIO3,Energia,0.4640,0.4680,0.9920
2,EMBR3,Indústria & Materiais,0.3960,0.4680,0.8450
3,ITUB4,Financeiro,0.1480,0.2690,0.5500
4,EGIE3,Energia,0.0800,0.1880,0.4240
5,GGBR4,Indústria & Materiais,0.0950,0.3560,0.2680
6,IBOV,Benchmark,0.0430,0.1980,0.2160
7,VALE3,Indústria & Materiais,0.0580,0.3390,0.1710
8,WEGE3,Indústria & Materiais,0.0120,0.3280,0.0360
9,ABEV3,Consumo,0.0090,0.2550,0.0340


## 5. P3 — Distribuição dos retornos diários
*Como se comportam estatisticamente os retornos (dispersão, assimetria e caudas)?*
- **Assimetria** negativa: quedas extremas mais frequentes que altas extremas.
- **Curtose** (excesso) alta: caudas pesadas, ou seja, mais dias extremos do que numa distribuição normal.

In [12]:
fig = px.histogram(
    acoes.dropna(subset=["retorno_diario"]).assign(ret_pct=lambda d: d["retorno_diario"] * 100),
    x="ret_pct", color="setor", marginal="box", nbins=120, opacity=0.75,
    title="P3 · Distribuição dos retornos diários por setor",
    labels={"ret_pct": "Retorno diário (%)", "setor": "Setor"},
    template=TEMA,
)
fig.update_layout(barmode="overlay", yaxis_title="Frequência (dias)")
fig.show()

In [13]:
resumo[["codigo", "retorno_diario_medio", "melhor_dia", "pior_dia", "assimetria", "curtose"]].sort_values(
    "curtose", ascending=False).round(4).reset_index(drop=True)

,codigo,retorno_diario_medio,melhor_dia,pior_dia,assimetria,curtose
0,BBDC4,0.0001,0.0691,-0.1738,-0.9909,8.9925
1,PETR4,0.0019,0.1103,-0.2071,-0.8662,7.6289
2,EMBR3,0.0018,0.1602,-0.1493,0.5886,4.2424
3,ABEV3,0.0002,0.0972,-0.0570,0.5309,3.4609
4,WEGE3,0.0003,0.0836,-0.1011,0.3922,2.3049
5,PRIO3,0.0019,0.1787,-0.1076,0.3032,2.1880
6,EGIE3,0.0004,0.0540,-0.0475,0.0295,1.6129
7,VALE3,0.0005,0.1040,-0.0759,0.3969,1.5057
8,ITUB4,0.0007,0.0827,-0.0728,0.2069,1.4803
9,RENT3,0.0003,0.0917,-0.0939,0.1776,1.2930


## 6. P4 — Drawdown (queda a partir do topo)
*Qual foi a maior queda percentual a partir do pico de cada ativo e quem foi mais resiliente?*

In [14]:
fig = px.area(
    diario.assign(dd_pct=diario["drawdown"] * 100), x="data", y="dd_pct", color="codigo",
    facet_col="codigo", facet_col_wrap=4,
    title="P4 · Drawdown ao longo do tempo (% abaixo do topo histórico)",
    labels={"data": "", "dd_pct": "Drawdown (%)"}, template=TEMA, height=750,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(showlegend=False)
fig.show()

In [15]:
dd = resumo.sort_values("drawdown_maximo", ascending=False)
fig = px.bar(
    dd.assign(dd_pct=dd["drawdown_maximo"] * 100), x="dd_pct", y="codigo", color="setor", orientation="h",
    title="P4 · Drawdown máximo por ativo (quanto menor a barra, mais resiliente)",
    labels={"dd_pct": "Drawdown máximo (%)", "codigo": "", "setor": "Setor"}, template=TEMA,
)
fig.update_yaxes(categoryorder="array", categoryarray=dd["codigo"].tolist()[::-1])
fig.show()
mais_resiliente = dd[dd["codigo"] != "IBOV"].iloc[0]
print(f"Mais resiliente: {mais_resiliente['codigo']} (drawdown máximo {mais_resiliente['drawdown_maximo']:.1%} "
      f"em {mais_resiliente['data_drawdown_maximo']:%d/%m/%Y})")

Mais resiliente: EGIE3 (drawdown máximo -17.7% em 05/08/2021)


## 7. P5 — Médias móveis (SMA 21 e SMA 50)
*Como o preço se comportou frente às médias de curto (21 pregões ≈ 1 mês) e médio prazo (50 pregões ≈ 2,5 meses)?*
Exemplo com o ativo de maior Sharpe; no app o usuário escolhe o ativo.

In [16]:
ATIVO = resumo[resumo["codigo"] != "IBOV"].sort_values("sharpe_empirico", ascending=False)["codigo"].iloc[0]
serie = diario[diario["codigo"] == ATIVO].melt(
    id_vars="data", value_vars=["fechamento_ajustado", "sma_21", "sma_50"], var_name="serie", value_name="preco")
serie["serie"] = serie["serie"].map({"fechamento_ajustado": "Fechamento ajustado", "sma_21": "SMA 21", "sma_50": "SMA 50"})
fig = px.line(
    serie, x="data", y="preco", color="serie",
    title=f"P5 · {ATIVO}: fechamento ajustado x médias móveis",
    labels={"data": "", "preco": "Preço ajustado (R$)", "serie": ""}, template=TEMA,
    color_discrete_map={"Fechamento ajustado": "#1F2937", "SMA 21": "#0B5394", "SMA 50": "#E69138"},
)
fig.show()

In [17]:
# Proporção de pregões em que o preço ficou ACIMA da SMA 50 (indicador simples de tendência)
tendencia = (acoes.dropna(subset=["sma_50"])
             .assign(acima=lambda d: d["fechamento_ajustado"] > d["sma_50"])
             .groupby("codigo")["acima"].mean().sort_values(ascending=False) * 100).round(1)
tendencia.rename("% dos pregões acima da SMA 50").to_frame()

,% dos pregões acima da SMA 50
codigo,
PETR4,72.3000
PRIO3,65.4000
EMBR3,57.6000
EGIE3,56.3000
ITUB4,56.2000
VALE3,54.2000
RENT3,53.7000
GGBR4,53.7000
BBDC4,49.6000


## 8. Extra — Correlação entre os retornos diários
O edital cita a **correlação setorial**. Correlação alta indica que os ativos tendem a se mover juntos (menor benefício de diversificação). **Correlação não implica causalidade.**

In [18]:
matriz = diario.pivot(index="data", columns="codigo", values="retorno_diario").corr()
fig = px.imshow(
    matriz, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
    title="Correlação de Pearson dos retornos diários (2021-2023)", template=TEMA, height=650,
)
fig.show()
pares = (matriz.where(np.triu(np.ones(matriz.shape, dtype=bool), k=1)).stack()
         .sort_values(ascending=False).rename("correlacao"))
print("Pares mais correlacionados:\n", pares.head(5).round(2).to_string())

Pares mais correlacionados:
 codigo  codigo
BBDC4   ITUB4    0.8100
IBOV    ITUB4    0.7100
        RENT3    0.7000
BBDC4   IBOV     0.6700
GGBR4   VALE3    0.6500


## 9. Achados preliminares (Pirâmide Invertida: contexto → insight → ação)

Os números abaixo são calculados ao vivo a partir dos dados.

In [19]:
r = resumo.set_index("codigo")
carteira = r.drop(index="IBOV")
melhor, pior = carteira["retorno_acumulado"].idxmax(), carteira["retorno_acumulado"].idxmin()
lider_sharpe = carteira["sharpe_empirico"].idxmax()
resiliente = carteira["drawdown_maximo"].idxmax()

print("CONTEXTO")
print(f"  12 ações de 4 setores, {diario['data'].min():%d/%m/%Y} a {diario['data'].max():%d/%m/%Y}; Ibovespa {r.loc['IBOV','retorno_acumulado']:+.1%}.")
print("INSIGHTS")
print(f"  P1: {len(acima)} de 12 ações superaram o Ibovespa; melhor {melhor} ({carteira.loc[melhor,'retorno_acumulado']:+.1%}), "
      f"pior {pior} ({carteira.loc[pior,'retorno_acumulado']:+.1%}).")
print(f"  P2: maior Sharpe empírico {lider_sharpe} ({carteira.loc[lider_sharpe,'sharpe_empirico']:.2f}).")
print(f"  P3: curtose mais alta {carteira['curtose'].idxmax()} ({carteira['curtose'].max():.1f}) -> caudas pesadas.")
print(f"  P4: mais resiliente {resiliente} (drawdown máx. {carteira.loc[resiliente,'drawdown_maximo']:.1%}); "
      f"maior queda {carteira['drawdown_maximo'].idxmin()} ({carteira['drawdown_maximo'].min():.1%}).")
print("AÇÃO (sugestão para o investidor, não recomendação)")
print("  Avaliar retorno sempre junto com volatilidade e drawdown: retorno alto não significou trajetória tranquila.")

CONTEXTO
  12 ações de 4 setores, 04/01/2021 a 28/12/2023; Ibovespa +13.2%.
INSIGHTS
  P1: 7 de 12 ações superaram o Ibovespa; melhor PETR4 (+228.7%), pior BBDC4 (-6.9%).
  P2: maior Sharpe empírico PETR4 (1.26).
  P3: curtose mais alta BBDC4 (9.0) -> caudas pesadas.
  P4: mais resiliente EGIE3 (drawdown máx. -17.7%); maior queda EMBR3 (-58.3%).
AÇÃO (sugestão para o investidor, não recomendação)
  Avaliar retorno sempre junto com volatilidade e drawdown: retorno alto não significou trajetória tranquila.


## 10. Limitações (o que **não** é possível inferir)
- **Período curto e específico** (3 anos, com ciclo de alta de juros e eleições): resultados passados não garantem resultados futuros.
- **Análise descritiva**: não identifica **causas** (ex.: não dá para afirmar que o setor de energia subiu *por causa* do petróleo sem análise econométrica).
- **Sharpe com taxa livre de risco = 0**: em um período de Selic alta, o Sharpe real (descontando o CDI) seria menor.
- **Volume financeiro estimado** (fechamento × quantidade), não é o valor oficial da B3.
- **Fonte Yahoo Finance**: 3 pregões sintéticos marcados; cisão XP/ITUB4 ajustada manualmente (fator documentado).
- **Universo de 12 ações** escolhido pelo edital: não representa todo o mercado.